## Peak Analysis (flexible range)
Same analysis as `PeakAnalysis.ipynb`, but the measurement positions are read from the file names in the folder
instead of being hard-coded to 5-200 cm. Any folder containing `<position>.pkl` files (e.g. `5.pkl`, `250.pkl`, `12.5.pkl`)
plus an optional `None.pkl` baseline will work, whatever the number of files or spacing between them.

## Load Data

In [ ]:
#folder = "Data/Pipe_Can_717_n2/"
#folder = "Data/PipeLarge_Target_717_10/"
folder = "Data/Water_Pipe/"
#folder = "Data/Air_Target2_717_0/"
#folder = "Data/Air_TargetNoMount_241_0/"

NONE_POSITION = 50.0  # actual position (cm) assigned to None.pkl; set to None to skip that file

# Set if the files are numbered backwards from this position: n.pkl is at REVERSE_FROM - n cm
# (Water_Pipe was recorded from 50cm inwards). Leave as None for normal datasets.
REVERSE_FROM = 50.0

# Leave as None to work these out from the folder name, or set to override
SENSOR = "717"   # Water_Pipe has no sensor in its folder name
RESOLUTION = None
THRESHOLD = None

In [ ]:
import pickle
import math
from pathlib import Path
from matplotlib import pyplot as plt
import numpy as np

In [ ]:
folderName = Path(folder).name
temp = folderName.split("_")
medium = temp[0]
target = temp[1] if len(temp) > 1 else "unknown target"
sensor = temp[2] if len(temp) > 2 else None
if SENSOR is not None:
    sensor = SENSOR
# Offset is stored as e.g. "10", "0" or "n2" (negative)
offsetStr = temp[3] if len(temp) > 3 else "0"
offset = -float(offsetStr[1:]) if offsetStr.startswith("n") else float(offsetStr)

if sensor == "717":
    resolution = 0.0352
    threshold = 100000 if medium == "Air" else 0.8e6
else:
    if sensor != "241":
        print(f"Unknown sensor '{sensor}', using 241 settings")
    resolution = 0.046875
    threshold = 300000 if medium == "Air" else 2.0e6

if RESOLUTION is not None:
    resolution = RESOLUTION
if THRESHOLD is not None:
    threshold = THRESHOLD

title = f"Measurement in {medium} using {target} as target with sensor at {offset:g}cm offset"
print(resolution)
print(threshold)
print(title)

In [ ]:
def loadFile(path):
    with open(path, 'rb') as f:
        fileData = pickle.load(f)
    fileData = [val[8] for val in fileData]
    return [sum(col) / len(col) for col in zip(*fileData)]

# Find every measurement file and work out its position from the file name
files = []
for path in Path(folder).glob("*.pkl"):
    if path.stem == "None":
        if NONE_POSITION is not None:
            files.append((NONE_POSITION, path))
        continue
    try:
        pos = float(path.stem)
    except ValueError:
        print(f"Skipping {path.name}: name is not a position")
        continue
    if REVERSE_FROM is not None:
        pos = REVERSE_FROM - pos
    files.append((pos, path))

files.sort(key=lambda f: f[0])
if not files:
    raise FileNotFoundError(f"No measurement files found in {folder}")

actualPos = np.array([pos for pos, _ in files])
print(f"Found {len(files)} files, positions {actualPos.min():g} to {actualPos.max():g} cm")

dataLin = []
dataLog = []
x = None
for pos, path in files:
    fileData = loadFile(path)
    if x is None:
        maxRange = len(fileData) * resolution
        x = np.linspace(0, maxRange, len(fileData))
    elif len(fileData) != len(x):
        raise ValueError(f"{path.name} has {len(fileData)} bins, expected {len(x)}")

    dataLin.append([(32.0/float(len(x))) * (2**(val/512.0)) for val in fileData])
    dataLog.append([(val*20.0*math.log10(2.0))/512.0 + 20.0*math.log10(32.0/float(len(x))) for val in fileData])

In [ ]:
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from sklearn.linear_model import RANSACRegressor
from sklearn.linear_model import LinearRegression

In [ ]:
measuredPos = []
measuredStrength = []
for i in range(len(dataLin)):
    smoothed = gaussian_filter1d(dataLin[i], sigma=1)
    peaks, properties = find_peaks(smoothed, height=threshold)
    if peaks.size:
        measuredPos.append(x[peaks[0]] * 100.0)
        measuredStrength.append(dataLog[i][peaks[0]])
    else:
        print(f"Not found at {actualPos[i]:g} cm")
        measuredPos.append(None)
        measuredStrength.append(None)

measuredPos = np.array(measuredPos)

In [ ]:
cleaned = [(x, y) for x, y in zip(actualPos, measuredPos)
           if x is not None and y is not None]
if len(cleaned) < 2:
    raise ValueError("Need at least 2 detected peaks to fit a line")

actual_clean, measured_clean = zip(*cleaned)

# Convert to numpy and reshape for sklearn
actual_clean = np.array(actual_clean).reshape(-1, 1)
measured_clean = np.array(measured_clean, dtype=float)

# RANSAC regression
ransac = RANSACRegressor(LinearRegression(), residual_threshold=10.0)
ransac.fit(actual_clean, measured_clean)

# Get slope and intercept
m = ransac.estimator_.coef_[0]
c = ransac.estimator_.intercept_

# Generate line of best fit
lineOBF = m * actualPos + c

inlier_mask = ransac.inlier_mask_
outlier_mask = ~inlier_mask

# Flatten actual_clean back to 1D for easier indexing
actual_clean_flat = actual_clean.flatten()

# Get inliers
actual_inliers = actual_clean_flat[inlier_mask]
measured_inliers = measured_clean[inlier_mask]

# Get outliers
actual_outliers = actual_clean_flat[outlier_mask]
measured_outliers = measured_clean[outlier_mask]

In [ ]:
plt.plot(actualPos, lineOBF, "b-", label="Line of Best Fit")
plt.xlabel("Actual Position (cm)")
plt.ylabel("Measured Position (cm)")
plt.text(
    0.05, 0.95, f"y={m:.2f}x + {c:.2f}",
    transform=plt.gca().transAxes,   # use axis coordinates (0-1)
    fontsize=10,
    verticalalignment='top',
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
)
x_err = 1
y_err = resolution*50
plt.errorbar(
    actual_inliers, measured_inliers,
    xerr=x_err, yerr=y_err,
    fmt='go', capsize=3, label="Inliers"
)

plt.errorbar(
    actual_outliers, measured_outliers,
    xerr=x_err, yerr=y_err,
    fmt='rx', capsize=3, label="Outliers"
)
plt.legend()
plt.grid()
plt.title(title)
plt.show()

## Inspect a single measurement

In [ ]:
inspectPos = 40    # actual position (cm) to look at; the closest file is used
i = int(np.argmin(np.abs(actualPos - inspectPos)))
print(f"Showing file at {actualPos[i]:g} cm")
smoothed = gaussian_filter1d(dataLin[i], sigma=1)
peaks, properties = find_peaks(smoothed, height=threshold)

plt.plot(x, dataLog[i])
plt.xlabel("Range (m)")
plt.ylabel("Signal (dB)")
plt.title(f"{title}\nTarget at {actualPos[i]:g} cm")
plt.show()